<a href="https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Yaya-1B Training — Google Colab

Trains Yaya-1B (~967M params) on OpenWebText with 8-bit AdamW.

**Requirements:** Runtime → Change runtime type → **T4 GPU** (or A100 for faster runs)

- bfloat16 + WSD schedule — stable training
- 8-bit AdamW (bitsandbytes) — cuts optimizer memory 8GB → 2GB
- Checkpoints saved to Google Drive — survive session resets
- Auto-resumes from latest checkpoint on re-run

## 1. Check GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('PyTorch:', torch.__version__)

## 2. Mount Google Drive (persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/yaya-checkpoints-1b'
SFT_CHECKPOINT_DIR = '/content/drive/MyDrive/yaya-sft-1b-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SFT_CHECKPOINT_DIR, exist_ok=True)
print('Pretrain checkpoints:', CHECKPOINT_DIR)
print('SFT checkpoints:     ', SFT_CHECKPOINT_DIR)

## 3. Clone repo + install dependencies

In [ ]:
import os
REPO_URL = 'https://github.com/jaylink-coder/miss-yaya.git'

if not os.path.exists('/content/miss-yaya'):
    !git clone {REPO_URL} /content/miss-yaya
else:
    !cd /content/miss-yaya && git pull

# bitsandbytes: 8-bit AdamW — cuts optimizer memory 8GB → 2GB
!pip install -q sentencepiece datasets pyyaml bitsandbytes

os.chdir('/content/miss-yaya/yaya-ai')
print('Working dir:', os.getcwd())

## 4. Download + tokenize OpenWebText
Uses 10% of OpenWebText (~80M tokens). Skipped on re-run if data already exists.

In [ ]:
import os, sys, numpy as np
sys.path.insert(0, '/content/miss-yaya')

TRAIN_DIR = 'data/processed/openwebtext/train'
EVAL_DIR  = 'data/processed/openwebtext/eval'
train_shard = os.path.join(TRAIN_DIR, 'shard_00000.bin')

if os.path.exists(train_shard) and os.path.getsize(train_shard) > 50_000_000:
    tokens_on_disk = os.path.getsize(train_shard) // 2
    print(f'Data already tokenized: {tokens_on_disk/1e6:.0f}M tokens — skipping.')
else:
    from datasets import load_dataset
    from src.tokenizer.tokenizer import YayaTokenizer

    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(EVAL_DIR,  exist_ok=True)

    print('Downloading OpenWebText (10% sample)...')
    ds = load_dataset('openwebtext', split='train[:10%]')
    print(f'Loaded {len(ds):,} documents')

    tokenizer = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    print('Tokenizer vocab size:', tokenizer.vocab_size)

    split = ds.train_test_split(test_size=0.01, seed=42)

    def tokenize_and_save(dataset, out_dir, fname):
        toks = []
        for i, row in enumerate(dataset):
            toks.extend(tokenizer.encode(row['text']))
            if (i + 1) % 10000 == 0:
                print(f'  {i+1:,} docs, {len(toks)/1e6:.0f}M tokens')
        arr = np.array(toks, dtype=np.uint16)
        arr.tofile(os.path.join(out_dir, fname))
        print(f'Saved {len(arr)/1e6:.0f}M tokens → {out_dir}/{fname}')
        return len(arr)

    print('Tokenizing train split...')
    n_train = tokenize_and_save(split['train'], TRAIN_DIR, 'shard_00000.bin')
    print('Tokenizing eval split...')
    n_eval  = tokenize_and_save(split['test'],  EVAL_DIR,  'eval.bin')
    print(f'\nDone: {n_train/1e6:.0f}M train tokens, {n_eval/1e6:.0f}M eval tokens')

## 5. Configure and train

In [ ]:
import yaml, glob

# Point training config at Drive checkpoints and OWT data
with open('configs/training/train_1b.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['checkpointing']['save_dir'] = CHECKPOINT_DIR
cfg['data']['train_data'] = TRAIN_DIR
cfg['data']['eval_data']  = EVAL_DIR
# Colab T4 has 15GB — tighten batch to be safe
cfg['training']['per_device_batch_size'] = 2
cfg['training']['gradient_accumulation_steps'] = 16

with open('configs/training/train_1b.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config updated.')

# Auto-resume from latest Drive checkpoint
ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
resume_flag = f'--resume {ckpts[-1]}' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else 'scratch')

In [ ]:
# Train! (disable W&B for clean output — remove the env vars to enable)
!WANDB_DISABLED=true WANDB_MODE=disabled python scripts/train.py \
    --model_config configs/model/yaya_1b.yaml \
    --train_config configs/training/train_1b.yaml \
    {resume_flag}

## 6. SFT — run after pretrain converges

In [ ]:
import yaml, glob

# Point SFT config at Drive checkpoints
with open('configs/training/sft_1b.yaml') as f:
    sft_cfg = yaml.safe_load(f)
sft_cfg['checkpointing']['save_dir'] = SFT_CHECKPOINT_DIR
with open('configs/training/sft_1b.yaml', 'w') as f:
    yaml.dump(sft_cfg, f)

# Get best pretrain checkpoint
ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
if not ckpts:
    print('ERROR: Run pretrain (cell above) first.')
else:
    pretrain_ckpt = ckpts[-1]
    sft_ckpts = sorted(glob.glob(f'{SFT_CHECKPOINT_DIR}/checkpoint-*'))
    if sft_ckpts:
        flags = f'--resume {sft_ckpts[-1]}'
        print('Resuming SFT from:', sft_ckpts[-1])
    else:
        flags = f'--pretrain_checkpoint {pretrain_ckpt}'
        print('Starting SFT from:', pretrain_ckpt)

    !WANDB_DISABLED=true WANDB_MODE=disabled python scripts/train_sft.py \
        --model_config configs/model/yaya_1b.yaml \
        --train_config configs/training/sft_1b.yaml \
        {flags}

## 7. Quick generation test

In [ ]:
import torch, glob
from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer
from src.inference.generator import TextGenerator
from src.training.checkpointing import CheckpointManager

model_config = load_model_config('configs/model/yaya_1b.yaml')
model = YayaForCausalLM(model_config)

# Load best SFT checkpoint (or pretrain if SFT not done yet)
sft_ckpts = sorted(glob.glob(f'{SFT_CHECKPOINT_DIR}/checkpoint-*'))
ckpt_dir = sft_ckpts[-1] if sft_ckpts else sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))[-1]
print('Loading checkpoint:', ckpt_dir)

ckpt_mgr = CheckpointManager(save_dir=os.path.dirname(ckpt_dir))
ckpt_mgr.load(model, checkpoint_path=ckpt_dir)

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

tokenizer = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
generator = TextGenerator(model, tokenizer, device=device)

prompt = 'Tell me something interesting about Africa.'
output = generator.generate(prompt, max_new_tokens=200, temperature=0.8, top_p=0.9)
print(output)